# Notebook 07: Temporal SHAP Analysis
Analyzing how feature importance changes over time (feature drift analysis)

## Cell 1 — Imports

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))
from src.data.data_loader import (
    load_config,
    load_dataset,
    sample_dataset
)
from src.data.preprocessing import (
    basic_preprocessing_pipeline
)
from src.features.feature_engineering import (
    feature_engineering_pipeline
)
from src.features.encoder import (
    encode_categorical_columns
)
from src.models.train_test_split import (
    temporal_train_test_split,
    split_features_target
)
from src.models.smote_pipeline import (
    apply_smote
)
from src.models.boosting_models import (
    train_xgboost
)
from src.visualization.temporal_shap import (
    compute_yearly_shap_importance,
    plot_temporal_feature_drift,
    get_top_features_by_year
)
from src.evaluation.drift_analysis import (
    compute_feature_drift
)

## Cell 2 — Load + Prepare Dataset

In [2]:
config = load_config()
df = load_dataset(config)
df = sample_dataset(df, config)
df = basic_preprocessing_pipeline(df)
df = feature_engineering_pipeline(df)
train_df, test_df = temporal_train_test_split(df)
X_train, X_test, y_train, y_test = split_features_target(
    train_df,
    test_df
)
X_train, X_test, encoders = encode_categorical_columns(
    X_train,
    X_test
)
X_train_smote, y_train_smote = apply_smote(
    X_train,
    y_train
)

print(f"✓ Data prepared successfully!")
print(f"  Training set (SMOTE): {X_train_smote.shape}")
print(f"  Test set: {X_test.shape}")


Loading dataset from:
/Users/nurnafisfuad/Desktop/AccidentXAI/data/raw/US_Accidents_March23.csv


Dataset loaded successfully.


Sampling 100000 rows...


Starting preprocessing pipeline...

Converting Start_Time to datetime format...
Removed 85 duplicate rows.

Selected relevant columns.

Dropping high-missing columns:

[]

Missing values handled successfully.

Preprocessing completed successfully.


Starting feature engineering pipeline...

Creating temporal features...

Creating rush-hour feature...

Creating night-driving feature...

Simplifying target variable...


Feature engineering completed.


Performing temporal split...

Train Shape: (71919, 30)
Test Shape: (16318, 30)

Separating features and target...

X_train shape: (71919, 29)
X_test shape: (16318, 29)

Encoding categorical columns...

Encoding completed.


Applying SMOTE...

Before SMOTE:

Counter({0: 54138, 1: 17781})

After SMOTE:

Counter({0: 54138, 1: 54138})
✓ Data prepared successfully!
  Training set (SMOTE): (1

## Cell 3 — Train XGBoost

In [3]:
xgb_model = train_xgboost(
    X_train_smote,
    y_train_smote
)

print("✓ XGBoost model trained successfully")


Training XGBoost...

XGBoost training completed.

✓ XGBoost model trained successfully


## Cell 4 — Feature Columns

In [4]:
feature_columns = X_train.columns.tolist()
feature_columns[:10]

['Temperature(F)',
 'Humidity(%)',
 'Pressure(in)',
 'Visibility(mi)',
 'Wind_Speed(mph)',
 'Weather_Condition',
 'Amenity',
 'Bump',
 'Crossing',
 'Give_Way']

## Cell 5 — Compute Yearly SHAP Importance

In [5]:
yearly_shap_df = compute_yearly_shap_importance(
    model=xgb_model,
    df=df,
    feature_columns=feature_columns,
    sample_size=300
)
yearly_shap_df.head()


Computing yearly SHAP importance...

Processing Year: 2020.0


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:Weather_Condition: str, Sunrise_Sunset: str, State: str

## Cell 6 — IMPORTANT RESEARCH INSIGHT

> **We use smaller yearly SHAP samples because:**
> 
> Temporal SHAP is computationally expensive.
> 
> Research practicality matters.
> 
> *"Better to have approximate answers than exact but late ones."*

## Cell 7 — Plot Temporal Feature Drift

In [ ]:
plot_temporal_feature_drift(
    yearly_shap_df,
    top_n=5
)

## Cell 8 — What You Should Observe

**Some features may remain stable.**

**Others may drift significantly.**

| Feature | Behavior |
|---------|----------|
| Night Driving | stable |
| Visibility | increasing |
| Weather | fluctuating |

---

**This becomes publishable insight.**

Understanding *why* features drift can lead to:
- Better model retraining strategies
- Domain-specific insights about changing conditions
- Improved feature engineering

## Cell 9 — Drift Analysis Table

In [ ]:
drift_df = compute_feature_drift(
    yearly_shap_df
)
drift_df.head(15)

## Cell 10 — Top Features By Year

In [ ]:
top_features = get_top_features_by_year(
    yearly_shap_df,
    top_n=5
)
top_features

## Optional Cell 11 — Drift Visualization Heatmap

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Pivot table for heatmap
pivot_df = yearly_shap_df.pivot(
    index='feature', 
    columns='year', 
    values='mean_shap'
)

# Plot heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(pivot_df.head(10), annot=True, fmt='.3f', cmap='RdBu_r', center=0)
plt.title('Feature Importance Heatmap Over Time (Top 10 Features)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Year')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## Optional Cell 12 — Stability Score Analysis

In [ ]:
# Calculate stability score (inverse of coefficient of variation)
stability_df = yearly_shap_df.groupby('feature')['mean_shap'].agg(['mean', 'std'])
stability_df['cv'] = stability_df['std'] / stability_df['mean']
stability_df['stability_score'] = 1 / stability_df['cv']
stability_df = stability_df.sort_values('stability_score', ascending=False)

print("Feature Stability Scores (higher = more stable):")
print(stability_df[['mean', 'std', 'stability_score']].head(10))

print("\n\nMost Unstable Features (potential drift):")
print(stability_df[['mean', 'std', 'stability_score']].tail(10))